# Analisi Esplorativa dei Dati: Approccio Settimanale 🇮🇹 [🇬🇧](02_eda_week_approach.ipynb)

## Previsione del Rischio di Infortunio nei Runner — Replica di Lovdal et al. (2021)

Questo notebook esplora il **dataset di carico di allenamento a livello settimanale**, il secondo approccio temporale utilizzato da Lovdal et al. per prevedere il rischio di infortunio nei runner agonisti.

L'approccio settimanale scambia **granularità temporale** con **aggregazione più ricca delle feature**: invece di 7 snapshot giornalieri di 10 feature raw, otteniamo **3 riepiloghi settimanali** di 22 feature aggregate (min, max, media delle metriche soggettive; totali di volume; conteggi sessioni) più **3 rapporti relativi km** che catturano i cambiamenti settimana-su-settimana del carico di allenamento.

**Differenza critica**: la variabile target è **continua** (da 0.0 a ~1.5+), rappresentando un punteggio di infortunio che deve essere **binarizzato** a una soglia scelta per la classificazione. La scelta di questa soglia influenza direttamente il bilanciamento delle classi e le performance del modello.

### Domande chiave

1. Com'è la **distribuzione del target continuo**?
2. Come influisce la **soglia di binarizzazione** sul bilanciamento delle classi?
3. Cosa rivelano i **rapporti relativi km** sui cambiamenti del carico di allenamento?
4. Come si confronta questo approccio con l'**approccio giornaliero** (Notebook 01)?

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

# Ensure src/ is importable when running from notebooks/
PROJECT_ROOT = Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.config import (
    ATHLETE_ID_COL,
    DATE_COL,
    INJURY_COL,
    N_WEEK_BLOCKS,
    RANDOM_SEED,
    WEEK_FEATURES,
    WEEK_INJURY_THRESHOLD,
    WEEK_RELATIVE_FEATURES,
)
from src.data_loading import get_feature_columns, load_day_data, load_week_data
from src.preprocessing.week_preprocessor import binarize_target
from src.utils.logging_config import setup_logging
from src.utils.plotting import INJURY_PALETTE, PALETTE, save_figure, set_style
from src.utils.reproducibility import set_global_seed

# --- Reproducibility & style ---
set_global_seed(RANDOM_SEED)
setup_logging()
set_style()

## 1. Caricamento Dati e Ispezione delle Dimensioni

In [ ]:
df_week = load_week_data()
feature_cols_week = get_feature_columns(df_week)

print(f"Shape: {df_week.shape[0]:,} rows × {df_week.shape[1]} columns")
print(f"Feature columns: {len(feature_cols_week)}")
print(f"  - Block features: {N_WEEK_BLOCKS} blocks × {len(WEEK_FEATURES)} features = {N_WEEK_BLOCKS * len(WEEK_FEATURES)}")
print(f"  - Relative ratios: {len(WEEK_RELATIVE_FEATURES)}")
print(f"Metadata columns: {ATHLETE_ID_COL}, {INJURY_COL}, {DATE_COL}")
print(f"\nUnique athletes: {df_week[ATHLETE_ID_COL].nunique()}")
print(f"Memory usage: {df_week.memory_usage(deep=True).sum() / 1e6:.1f} MB")

In [ ]:
df_week.head()

### Nota sulla struttura dati

Le 22 feature settimanali catturano informazioni più ricche rispetto alle 10 feature giornaliere:

- **Conteggi**: `nr_sessions`, `nr_rest_days`, `nr_tough_sessions`, `nr_days_with_interval_session`, `nr_strength_trainings`
- **Totali di volume**: `total_kms`, `total_km_z3_4`, `total_km_z5_t1_t2`, `total_km_z3_z4_z5_t1_t2`, `total_hours_alternative_training`
- **Massimi**: `max_km_one_day`, `max_km_z3_4_one_day`, `max_km_z5_t1_t2_one_day`
- **Riepiloghi soggettivi**: `avg/min/max` di `exertion`, `training_success`, `recovery`

Le **triple min/max/avg** sono particolarmente interessanti — catturano la **variazione intra-settimanale** che l'approccio giornaliero non coglie. Un atleta con `avg_exertion = 5` e `max_exertion = 9` ha avuto una settimana molto diversa da uno con `avg_exertion = 5` e `max_exertion = 6`.

---

## 2. Distribuzione del Target Continuo

A differenza dell'approccio giornaliero (binario 0/1), l'approccio settimanale ha un **punteggio di infortunio continuo**. Comprendere questa distribuzione è essenziale prima di scegliere una soglia di binarizzazione.

In [ ]:
injury_scores = df_week[INJURY_COL]

print("Continuous injury score statistics:\n")
print(f"  Mean:       {injury_scores.mean():.4f}")
print(f"  Median:     {injury_scores.median():.4f}")
print(f"  Std:        {injury_scores.std():.4f}")
print(f"  Min:        {injury_scores.min():.4f}")
print(f"  Max:        {injury_scores.max():.4f}")
print("\nPercentiles:")
for p in [25, 50, 75, 90, 95, 99]:
    print(f"  {p}th: {injury_scores.quantile(p / 100):.4f}")
print(f"\nExactly 0.0: {(injury_scores == 0).sum():,} ({(injury_scores == 0).mean() * 100:.2f}%)")
print(f"Above 0.5:   {(injury_scores >= 0.5).sum():,} ({(injury_scores >= 0.5).mean() * 100:.2f}%)")

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))

ax.hist(injury_scores, bins=100, color=PALETTE["primary"], alpha=0.7, edgecolor="white")

# Threshold lines
thresholds = {0.1: ("dotted", PALETTE["neutral"]), 0.25: ("dashed", PALETTE["secondary"]), 0.5: ("solid", PALETTE["negative"])}
for t, (ls, color) in thresholds.items():
    ax.axvline(t, linestyle=ls, color=color, linewidth=2, label=f"Threshold = {t}")

ax.set_xlabel("Injury score (continuous)")
ax.set_ylabel("Count")
ax.set_title("Continuous Target Distribution — Week Approach", fontweight="bold")
ax.legend()
sns.despine()

save_figure(fig, "02_continuous_target_distribution", subdir="eda", close=False)
plt.show()
plt.close(fig)

### Interpretazione

La distribuzione del target continuo è **fortemente asimmetrica a destra** con un massiccio picco a 0.0 — la grande maggioranza delle settimane non mostra segnale di infortunio.

La natura continua fornisce **più informazione** rispetto al target binario giornaliero: invece di un secco sì/no, vediamo gradazioni di rischio infortunio. Valori prossimi a 0 suggeriscono rischio minimo, mentre valori superiori a 0.5 indicano un coinvolgimento significativo dell'infortunio.

**La scelta della soglia conta**: dove tracciamo la linea binaria determina il bilanciamento delle classi e, di conseguenza, ciò che il modello impara a rilevare. La prossima sezione quantifica questo compromesso.

---

## 3. Sensibilità alla Soglia: Come la Binarizzazione Cambia il Problema

L'articolo usa **0.5** come soglia di binarizzazione (ADR-002). Ma questa scelta non è ovvia — soglie più basse catturano segnali di infortunio più lievi ma aumentano la dimensione della classe positiva. Soglie più alte si concentrano su infortuni più gravi ma riducono il numero di campioni positivi.

Questa analisi documenta come il bilanciamento delle classi cambia al variare delle soglie.

In [ ]:
thresholds = [0.1, 0.25, 0.5, 0.75, 1.0]

sensitivity_data: list[dict[str, float]] = []
for t in thresholds:
    y_bin = binarize_target(injury_scores, threshold=t)
    n_pos = int(y_bin.sum())
    n_total = len(y_bin)
    pct_pos = n_pos / n_total * 100
    ratio = (n_total - n_pos) / n_pos if n_pos > 0 else float("inf")
    sensitivity_data.append({
        "threshold": t,
        "n_positive": n_pos,
        "pct_positive": pct_pos,
        "neg_pos_ratio": ratio,
    })

sensitivity_df = pd.DataFrame(sensitivity_data)
print("Threshold sensitivity analysis:\n")
print(sensitivity_df.to_string(index=False, float_format="{:.2f}".format))

In [ ]:
fig, ax1 = plt.subplots(figsize=(10, 5))

color1 = PALETTE["primary"]
ax1.plot(
    sensitivity_df["threshold"],
    sensitivity_df["pct_positive"],
    color=color1,
    marker="o",
    linewidth=2,
    markersize=8,
    label="Positive class %",
)
ax1.set_xlabel("Binarization threshold")
ax1.set_ylabel("Positive class (%)", color=color1)
ax1.tick_params(axis="y", labelcolor=color1)

# Annotate each point
for _, row in sensitivity_df.iterrows():
    ax1.annotate(
        f"{row['pct_positive']:.1f}%",
        (row["threshold"], row["pct_positive"]),
        textcoords="offset points",
        xytext=(0, 12),
        ha="center",
        fontsize=9,
        fontweight="bold",
        color=color1,
    )

ax2 = ax1.twinx()
color2 = PALETTE["negative"]
ax2.plot(
    sensitivity_df["threshold"],
    sensitivity_df["neg_pos_ratio"],
    color=color2,
    marker="s",
    linewidth=2,
    markersize=8,
    linestyle="--",
    label="Neg:Pos ratio",
)
ax2.set_ylabel("Negative:Positive ratio", color=color2)
ax2.tick_params(axis="y", labelcolor=color2)

# Highlight the paper's threshold
ax1.axvline(0.5, color=PALETTE["neutral"], linestyle=":", linewidth=1.5, alpha=0.7)
ax1.text(0.52, ax1.get_ylim()[1] * 0.9, "Paper's\nthreshold", fontsize=9, color=PALETTE["neutral"])

# Combined legend
lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, loc="center right")

ax1.set_title("Threshold Sensitivity — Effect on Class Balance", fontweight="bold")
fig.tight_layout()

save_figure(fig, "02_threshold_sensitivity", subdir="eda", close=False)
plt.show()
plt.close(fig)

### Interpretazione

L'analisi di sensibilità alla soglia rivela il **compromesso fondamentale** nella binarizzazione di un target continuo:

- **Alla soglia 0.1**: più osservazioni vengono etichettate come "infortunate", creando uno sbilanciamento meno estremo. Ma molte di queste potrebbero essere **segnali lievi** — fastidi minori o report precauzionali, non infortuni reali.

- **Alla soglia 0.5 (scelta dell'articolo)**: una definizione più conservativa di infortunio. Lo sbilanciamento è meno estremo rispetto all'1.36% dell'approccio giornaliero, rendendo il problema leggermente più facile da modellare.

- **Alla soglia 1.0**: solo gli infortuni più gravi vengono catturati, potenzialmente troppo pochi perché il modello possa apprendere.

**Perché 0.5?** L'articolo ha scelto questa soglia come equilibrio tra sensibilità (catturare infortuni reali) e specificità (non etichettare problemi minori come infortuni). Seguiamo la loro convenzione per consentire il confronto diretto dei risultati (ADR-002).

---

## 4. Rapporti Relativi Km: Cambiamenti del Carico Settimana su Settimana

Le tre feature di rapporto sono la **risorsa unica dell'approccio settimanale**:
- `rel_total_kms_week_0_1`: km Settimana 0 / km Settimana 1
- `rel_total_kms_week_0_2`: km Settimana 0 / km Settimana 2
- `rel_total_kms_week_1_2`: km Settimana 1 / km Settimana 2

Un rapporto **> 1.0** significa che la settimana al numeratore ha avuto un volume maggiore (spike di allenamento). Un rapporto **< 1.0** significa che il volume è diminuito (scarico).

Queste feature codificano direttamente il concetto di **rapporto acuto:cronico del carico (ACWR)** dalla scienza dello sport — uno dei predittori più forti del rischio di infortunio.

In [ ]:
# Binarize target at paper threshold for this analysis
df_week["injury_binary"] = binarize_target(df_week[INJURY_COL], threshold=WEEK_INJURY_THRESHOLD)

print("Ratio feature statistics:\n")
for feat in WEEK_RELATIVE_FEATURES:
    series = df_week[feat]
    vals = series.dropna()
    zero_or_nan_pct = ((series == 0) | series.isna()).mean() * 100
    print(f"{feat}:")
    print(f"  Mean: {vals.mean():.3f}, Median: {vals.median():.3f}, Std: {vals.std():.3f}")
    print(f"  Min: {vals.min():.3f}, Max: {vals.max():.3f}")
    print(f"  % above 1.0 (load increase): {(vals > 1.0).mean() * 100:.1f}%")
    print(f"  % exactly 0.0 or NaN: {zero_or_nan_pct:.1f}%")
    print()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

short_names = ["Week 0/1", "Week 0/2", "Week 1/2"]

for ax, feat, name in zip(axes, WEEK_RELATIVE_FEATURES, short_names):
    for label, color, lbl in [
        (0, INJURY_PALETTE[0], "No Injury"),
        (1, INJURY_PALETTE[1], "Injury"),
    ]:
        subset = df_week[df_week["injury_binary"] == label][feat].dropna()
        ax.hist(
            subset,
            bins=50,
            alpha=0.5,
            color=color,
            label=f"{lbl} (n={len(subset):,})",
            density=True,
            edgecolor="white",
        )

    ax.axvline(1.0, color="black", linestyle="--", linewidth=1.2, label="Ratio = 1.0 (equal load)")
    ax.set_xlabel("Km ratio")
    ax.set_ylabel("Density")
    ax.set_title(f"Relative Km Ratio — {name}", fontweight="bold")
    ax.legend(fontsize=8)
    sns.despine(ax=ax)

fig.suptitle("Week-over-Week Km Ratio Distributions by Injury Status", fontweight="bold", fontsize=14, y=1.03)
fig.tight_layout()

save_figure(fig, "02_ratio_distributions", subdir="eda", close=False)
plt.show()
plt.close(fig)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for ax, feat, name in zip(axes, WEEK_RELATIVE_FEATURES, short_names):
    # Use hexbin for dense scatter
    valid = df_week[[feat, INJURY_COL]].dropna()
    hb = ax.hexbin(
        valid[feat],
        valid[INJURY_COL],
        gridsize=40,
        cmap="Blues",
        mincnt=1,
    )

    ax.axhline(WEEK_INJURY_THRESHOLD, color=PALETTE["negative"], linestyle="--", linewidth=1.5, label=f"Threshold = {WEEK_INJURY_THRESHOLD}")
    ax.axvline(1.0, color=PALETTE["neutral"], linestyle=":", linewidth=1.2, alpha=0.7)
    ax.set_xlabel(f"Km ratio — {name}")
    ax.set_ylabel("Injury score (continuous)")
    ax.set_title(f"Ratio vs Injury — {name}", fontweight="bold")
    ax.legend(fontsize=8)

fig.suptitle("Km Ratios vs Continuous Injury Score", fontweight="bold", fontsize=14, y=1.03)
fig.tight_layout()

save_figure(fig, "02_ratio_vs_injury", subdir="eda", close=False)
plt.show()
plt.close(fig)

### Interpretazione: Il Rapporto Acuto:Cronico del Carico

Le feature di rapporto codificano il **concetto ACWR** dalla scienza dello sport:

- **Rapporto > 1.0 = spike di allenamento**: l'atleta ha aumentato il volume di allenamento rispetto alla settimana di riferimento. Questa è la "zona di pericolo" identificata da Blanch e Gabbett (2016): incrementi rapidi del carico (ACWR > 1.5) sono associati a un **rischio di infortunio 2–4 volte superiore**.

- **Rapporto < 1.0 = scarico**: l'atleta ha ridotto il volume. Sebbene generalmente protettivo, rapporti molto bassi (< 0.5) possono indicare deallenamento, che a sua volta aumenta il rischio di infortunio alla ripresa dell'allenamento normale.

- **Rapporto ≈ 1.0 = carico stabile**: allenamento costante settimana su settimana — la zona più sicura.

**Cosa cercare negli istogrammi:**
- Se il **gruppo infortunati** mostra una distribuzione più ampia (rapporti più estremi su entrambi i lati), conferma che la variabilità del carico — sia gli spike che i cali — è un fattore di rischio.
- Se il gruppo infortunati è spostato a destra (> 1.0), punta specificamente agli spike di carico.

Questo è il **differenziatore dell'expertise di dominio** di questa analisi: non stiamo solo calcolando statistiche sulle feature — le stiamo interpretando attraverso la lente della teoria di gestione del carico di allenamento.

---

## 5. Distribuzioni delle Feature Settimanali (Settimana 0)

In [ ]:
week0_cols = [f"week_0_{feat}" for feat in WEEK_FEATURES]

n_feats = len(WEEK_FEATURES)
n_cols = 4
n_rows = (n_feats + n_cols - 1) // n_cols  # ceiling division

fig, axes = plt.subplots(n_rows, n_cols, figsize=(20, n_rows * 3.5))
axes = axes.flatten()

for i, (col, feat_name) in enumerate(zip(week0_cols, WEEK_FEATURES)):
    ax = axes[i]
    data = df_week[col].dropna()
    ax.hist(data, bins=50, color=PALETTE["primary"], alpha=0.7, edgecolor="white")
    median_val = data.median()
    ax.axvline(median_val, color=PALETTE["negative"], linestyle="--", linewidth=1,
               label=f"median={median_val:.2f}")
    ax.set_title(feat_name, fontweight="bold", fontsize=10)
    ax.legend(fontsize=7)

# Hide unused subplots
for j in range(n_feats, len(axes)):
    axes[j].set_visible(False)

fig.suptitle("Feature Distributions — Week 0 (Most Recent Week)", fontweight="bold", fontsize=14, y=1.01)
fig.tight_layout()

save_figure(fig, "02_week0_feature_distributions", subdir="eda", close=False)
plt.show()
plt.close(fig)

### Note sulle distribuzioni

Differenze chiave rispetto all'approccio giornaliero:

- **Meno zero-inflate**: gli aggregati settimanali contengono quasi sempre qualche allenamento (anche se singoli giorni erano di riposo). `nr_rest_days` cattura il pattern di riposo esplicitamente.

- **Triple min/max/avg**: le metriche soggettive (`exertion`, `training_success`, `recovery`) ora hanno tre colonne ciascuna — `avg`, `min`, `max`. Questo cattura la **variazione intra-settimanale**: una settimana con una sessione molto dura si manifesta come un divario tra `avg_exertion` e `max_exertion`.

- **Feature di conteggio**: `nr_sessions`, `nr_tough_sessions`, `nr_days_with_interval_session` sono interi discreti, che mostrano i pattern tipici di allenamento (es. la maggior parte degli atleti fa 1–2 sessioni di intervalli a settimana).

---

## 6. Confronto Approccio Giornaliero vs. Settimanale

I due approcci rappresentano visioni fondamentalmente diverse degli **stessi dati di allenamento sottostanti**. Comprendere le loro differenze informa quale approccio potrebbe essere più efficace per la previsione.

In [ ]:
df_day = load_day_data()

day_positive_pct = df_day[INJURY_COL].mean() * 100
week_positive_pct = (binarize_target(df_week[INJURY_COL], threshold=WEEK_INJURY_THRESHOLD)).mean() * 100

comparison = pd.DataFrame({
    "Aspect": [
        "Rows",
        "Total columns",
        "Feature columns",
        "Temporal blocks",
        "Features per block",
        "Unique features",
        "Target type",
        "Positive class %",
        "Sentinel values",
    ],
    "Day Approach": [
        f"{df_day.shape[0]:,}",
        str(df_day.shape[1]),
        str(len(get_feature_columns(df_day))),
        "7 (days)",
        "10",
        "Raw daily metrics",
        "Binary (0/1)",
        f"{day_positive_pct:.2f}%",
        "Yes (−0.01)",
    ],
    "Week Approach": [
        f"{df_week.shape[0]:,}",
        str(df_week.shape[1]),
        f"{len(feature_cols_week)} ({N_WEEK_BLOCKS}×{len(WEEK_FEATURES)} + {len(WEEK_RELATIVE_FEATURES)} ratios)",
        "3 (weeks) + 3 ratios",
        "22",
        "Aggregated + ratios",
        f"Continuous (binarized at {WEEK_INJURY_THRESHOLD})",
        f"{week_positive_pct:.2f}%",
        "No",
    ],
})

print(comparison.to_string(index=False))

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 4))

# Day approach: bar chart
day_counts = df_day[INJURY_COL].value_counts()
day_labels = ["No Injury (0)", "Injury (1)"]
ax1.barh(day_labels, [day_counts[0], day_counts[1]], color=[INJURY_PALETTE[0], INJURY_PALETTE[1]], height=0.5)
ax1.set_xlabel("Count")
ax1.set_title(f"Day Approach — Binary Target\n({day_positive_pct:.2f}% positive)", fontweight="bold")
for i, (count, label) in enumerate(zip([day_counts[0], day_counts[1]], day_labels)):
    ax1.text(count + max(day_counts) * 0.01, i, f"n={count:,}", va="center", fontweight="bold", fontsize=9)
ax1.set_xlim(0, max(day_counts) * 1.15)
sns.despine(ax=ax1, left=True)

# Week approach: histogram
ax2.hist(df_week[INJURY_COL], bins=80, color=PALETTE["primary"], alpha=0.7, edgecolor="white")
ax2.axvline(WEEK_INJURY_THRESHOLD, color=PALETTE["negative"], linestyle="--", linewidth=2,
            label=f"Threshold = {WEEK_INJURY_THRESHOLD} → {week_positive_pct:.2f}% positive")
ax2.set_xlabel("Injury score")
ax2.set_ylabel("Count")
ax2.set_title("Week Approach — Continuous Target", fontweight="bold")
ax2.legend(fontsize=9)
sns.despine(ax=ax2)

fig.suptitle("Target Distribution Comparison: Day vs. Week", fontweight="bold", fontsize=14, y=1.05)
fig.tight_layout()

save_figure(fig, "02_day_vs_week_targets", subdir="eda", close=False)
plt.show()
plt.close(fig)

### Interpretazione

I due approcci rappresentano un **compromesso risoluzione vs. ricchezza**:

| | Approccio Giornaliero | Approccio Settimanale |
|---|---|---|
| **Risoluzione temporale** | Alta (giornaliera) | Più bassa (settimanale) |
| **Ricchezza delle feature** | Base (10 metriche raw) | Ricca (22 aggregate + rapporti) |
| **Sbilanciamento classi** | Estremo (~1.36%) | Meno estremo (dipende dalla soglia) |
| **Gestione sentinel** | Necessaria | Non necessaria |
| **Vantaggio unico** | Pattern giornalieri, variazione giorno-per-giorno | Variazione intra-settimanale, rapporti ACWR |

**Risultato controintuitivo dell'articolo**: nonostante abbia meno feature e più semplici, l'**approccio giornaliero ha performato meglio** (AUC-ROC 0.724 vs. 0.678). Questo suggerisce che la **granularità temporale giornaliera cattura pattern rilevanti per l'infortunio** — come uno spike improvviso in un singolo giorno — che l'aggregazione settimanale attenua.

In termini di scienza dello sport: un atleta potrebbe avere un total_km settimanale "normale" ma un giorno estremamente intenso all'interno di quella settimana. L'approccio giornaliero cattura questa anomalia direttamente; l'approccio settimanale la diluisce in una media.

---

## 7. Insight Chiave e Implicazioni

### Riepilogo dei risultati

1. **Il target continuo è fortemente zero-inflato**: la grande maggioranza delle settimane mostra zero segnale di infortunio. La binarizzazione a 0.5 produce uno sbilanciamento meno estremo rispetto all'approccio giornaliero, ma il problema resta sbilanciato.

2. **La scelta della soglia influisce significativamente sul bilanciamento delle classi**: passando da 0.5 a 0.25 si raddoppia approssimativamente la classe positiva. L'analisi di sensibilità giustifica la scelta dell'articolo di 0.5 documentando al contempo alternative per sperimentazioni future (ADR-002).

3. **Le feature di rapporto codificano i cambiamenti acuto:cronico del carico**: i rapporti relativi km sono la risorsa predittiva unica dell'approccio settimanale, codificando direttamente spike e cali del carico di allenamento — il singolo predittore più forte dell'infortunio nella letteratura di scienza dello sport.

4. **Feature più ricche, risoluzione più grossolana**: 22 feature per settimana (con triple min/max/avg) vs. 10 feature giornaliere raw. La codifica min/max/avg cattura la variazione intra-settimanale ma perde la granularità giornaliera.

5. **Nessun valore sentinel nei dati settimanali**: l'aggregazione settimanale elimina naturalmente il problema della codifica dei giorni di riposo — i giorni di riposo sono catturati da `nr_rest_days` e da valori bassi nelle altre feature, anziché da valori sentinel.

### Prossimi passi

→ **Notebook 03**: Implementare il preprocessing per entrambi gli approcci — sostituzione sentinel (giornaliero), binarizzazione del target (settimanale), scaling e split train/test a livello di atleta.

→ **Notebook 04–05**: Addestramento e valutazione dei modelli, dove verificheremo se il vantaggio di AUC dell'approccio giornaliero si mantiene nella nostra replica.

In [ ]:
# Clean up temporary column
df_week.drop(columns=["injury_binary"], inplace=True)